# Three-way kernel-latency comparison v2 — QAT vs PTQ vs FP16

Honest, raw analysis of the trtexec `--exportProfile` JSONs. **Every kernel is a real fused
TensorRT kernel** — `QuantizeLinear` appears *folded into* conv names, never as a standalone
'Q/DQ pair'. We use the raw names as-is; the only standalone quantize kernels are the
`__myl_MulMinMax`/`CastMul` activation-quantize ops (categorised 'activation quant').
**Analysis only** — the three JSONs are read-only inputs.


## STEP 1 — load (medianMs throughout; skip the `{count}` header)

In [ ]:
import json, re
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 400); pd.set_option('display.max_colwidth', 90)

KJ = Path('../Per_kernel_json_file')
FILES = {'QAT':'qat_batch32_per_kernel_profile.json',
         'PTQ':'ptq_int8_per_kernel_profile.json',
         'FP16':'fp16_per_kernel_profile.json'}
ANCHOR = {'QAT':1.200, 'PTQ':1.070, 'FP16':1.127}   # measured whole-engine medians (exclusive A100)
COLORS = {'QAT':'#c0392b', 'PTQ':'#2980b9', 'FP16':'#27ae60'}

def load_profile(path):
    raw = json.load(open(path))
    rows = [e for e in raw if 'name' in e]          # element [0] is {'count': N}
    df = pd.DataFrame(rows)[['name','averageMs','medianMs','percentage']]
    for c in ['averageMs','medianMs','percentage']: df[c] = pd.to_numeric(df[c])
    return df

raw_df = {t: load_profile(KJ/FILES[t]) for t in FILES}
for t, df in raw_df.items():
    print(f'{t:5} kernels={len(df):4d}  sum(medianMs)={df["medianMs"].sum():.4f} ms  '
          f'(whole-engine measured {ANCHOR[t]} ms)')


## STEP 2 — parse the RAW name: layer, subpath, op_type
`reformat/copy` is checked first because those names contain 'Conv'. The QuantizeLinear inside
a fused conv kernel is **folded in** — the kernel is 'conv+SiLU fused', its latency is the whole
fused kernel (not a separable Q cost).

In [ ]:
def op_type(n):
    # reformat/copy FIRST (these names contain 'Conv'; ' copy' suffix has a space)
    if 'Reformatting' in n or 'CopyNode' in n or 'copy' in n: return 'reformat/copy'
    has_conv = 'Conv' in n; has_pwn = 'PWN' in n
    if has_conv and has_pwn and ('Sigmoid' in n or 'Mul' in n): return 'conv+SiLU fused'
    if has_conv and not has_pwn: return 'conv only'
    if n.startswith('PWN(') and 'Sigmoid' in n and 'Mul' in n and 'Conv' not in n: return 'standalone SiLU'
    if n.startswith('__myl_') and ('MulMinMax' in n or 'CastMul' in n): return 'activation quant'
    if any(k in n for k in ['MatMul','MaxrSub','DivMul','MoveSlicSlicTran','MulAddAdd','/attn/']): return 'attention'
    if any(k in n for k in ['Topk','ReshReshResh','ForeignNode','Gather']): return 'NMS/decode'
    if n.startswith('__myl_') and ('Resh' in n or 'Move' in n or 'Tran' in n) and 'MulMinMax' not in n: return 'reshape/move'
    if 'MaxPool' in n: return 'maxpool'
    if 'Resize' in n or 'Concat' in n or 'Slice_output' in n or 'Split' in n: return 'resize/concat'
    return 'other'

def parse(name):
    m = re.search(r'model[./](\d+)', name)
    layer = f'model.{m.group(1)}' if m else 'other/glue'
    subpath = ''
    if m:
        after = name[m.end():]
        s = re.search(r'[./]([\w.]+?)(?:/conv|/act|/weight|/Conv|\b)', after)
        subpath = s.group(1) if s else ''
    return layer, subpath, op_type(name)

def build_df(tag):
    df = load_profile(KJ/FILES[tag])
    p = df['name'].apply(lambda x: pd.Series(parse(x), index=['layer','subpath','op_type']))
    return pd.concat([df, p], axis=1)

DFS = {t: build_df(t) for t in FILES}
# sanity: 'other' must be tiny (only genuinely-uncategorisable edge kernels)
for t in FILES: print(t, "'other' kernels:", int((DFS[t]['op_type']=='other').sum()))


## STEP 3 — per-file kernel table (sorted by layer, then medianMs desc) + CSV with TOTAL row

In [ ]:
def per_file_table(tag):
    d = DFS[tag].copy()
    d['name80'] = d['name'].str.slice(0, 80)
    d = d[['layer','subpath','op_type','medianMs','name80']].sort_values(
            ['layer','medianMs'], ascending=[True, False]).reset_index(drop=True)
    total = pd.DataFrame([{'layer':'TOTAL','subpath':'','op_type':f'{len(d)} kernels',
                           'medianMs':d['medianMs'].sum(),'name80':''}])
    out = pd.concat([d, total], ignore_index=True)
    out.to_csv(f'kernels_{tag}.csv', index=False)
    return out

tbl_QAT  = per_file_table('QAT')
tbl_PTQ  = per_file_table('PTQ')
tbl_FP16 = per_file_table('FP16')
print('saved kernels_QAT.csv / kernels_PTQ.csv / kernels_FP16.csv')
tbl_QAT.head(20)   # preview; swap to tbl_PTQ / tbl_FP16


## STEP 4 — common layers (present in ALL 3), summed median ms + deltas

In [ ]:
common = sorted(set.intersection(*[set(DFS[t]['layer']) for t in FILES]))
rows = {t: DFS[t][DFS[t]['layer'].isin(common)].groupby('layer')['medianMs'].sum() for t in FILES}
comp = pd.DataFrame(rows).reindex(common).fillna(0)
comp['QAT-PTQ']  = comp['QAT'] - comp['PTQ']
comp['QAT-FP16'] = comp['QAT'] - comp['FP16']
comp = comp.sort_values('QAT', ascending=False)
total = pd.DataFrame(comp.sum()).T; total.index = ['TOTAL']
comp_out = pd.concat([comp, total])
comp_out.round(4).to_csv('layer_comparison_common.csv')
print(f'{len(common)} common layers'); comp_out.round(4)


## STEP 5 — grouped bar: per-layer summed median latency (common layers)

In [ ]:
lay = comp.index.tolist(); x = np.arange(len(lay)); w = 0.27
fig, ax = plt.subplots(figsize=(13, 4.8))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, comp[t], w, label=t, color=COLORS[t])
ax.set_xticks(x); ax.set_xticklabels(lay, rotation=45, ha='right')
ax.set_ylabel('summed median latency (ms)')
ax.set_title('Per-layer summed median latency — QAT vs PTQ vs FP16 (common layers)')
ax.legend(); plt.tight_layout(); plt.savefig('layer_comparison.png', dpi=130); plt.show()


## STEP 6 — per-engine op-type summary (sum ms + count) + CSV with TOTAL row

In [ ]:
def op_summary(tag):
    d = DFS[tag].groupby('op_type').agg(sum_ms=('medianMs','sum'), n_kernels=('medianMs','size'))
    d = d.sort_values('sum_ms', ascending=False)
    total = pd.DataFrame({'sum_ms':[d['sum_ms'].sum()], 'n_kernels':[int(d['n_kernels'].sum())]}, index=['TOTAL'])
    out = pd.concat([d, total]); out.round(4).to_csv(f'op_summary_{tag}.csv')
    return out

for t in FILES:
    print(f'--- {t} ---'); display(op_summary(t).round(4))


## STEP 7 — op-type histograms (per engine) + combined grouped bar

In [ ]:
op_ms = pd.DataFrame({t: DFS[t].groupby('op_type')['medianMs'].sum() for t in FILES}).fillna(0)
order = op_ms.sum(axis=1).sort_values(ascending=False).index.tolist()
op_ms = op_ms.reindex(order)

# per-engine horizontal bars
for t in FILES:
    s = op_ms[t][op_ms[t] > 0].sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(s.index, s.values, color=COLORS[t])
    for i, v in enumerate(s.values): ax.text(v+0.003, i, f'{v:.3f}', va='center', fontsize=8)
    ax.set_xlabel('summed median ms'); ax.set_title(f'{t} — latency by op-type')
    ax.set_xlim(0, op_ms.values.max()*1.15)
    plt.tight_layout(); plt.savefig(f'optype_{t}.png', dpi=130); plt.show()

# combined grouped bar
x = np.arange(len(op_ms)); w = 0.27
fig, ax = plt.subplots(figsize=(13, 5))
for i, t in enumerate(['QAT','PTQ','FP16']):
    ax.bar(x+(i-1)*w, op_ms[t], w, label=t, color=COLORS[t])
ax.set_xticks(x); ax.set_xticklabels(op_ms.index, rotation=40, ha='right')
ax.set_ylabel('summed median ms'); ax.set_title('Latency by op-type — QAT vs PTQ vs FP16')
ax.legend(); plt.tight_layout(); plt.savefig('optype_combined.png', dpi=130); plt.show()


## STEP 8 — sanity: per-kernel median sums are inflated by CUDA-event instrumentation

In [ ]:
recon = pd.DataFrame({
    'kernels':       {t: len(DFS[t]) for t in FILES},
    'sum_medianMs':  {t: DFS[t]['medianMs'].sum() for t in FILES},
    'measured_ms':   ANCHOR,
})
recon['inflation_x'] = (recon['sum_medianMs']/recon['measured_ms']).round(2)
display(recon.round(4))
print('Per-kernel median sums are ~1.4-1.6x the true whole-engine median (CUDA-event overhead).')
print('Use RELATIVE comparisons across kernels/engines; absolute per-kernel ms are diagnostic only.')


# STEP 9 — Quantified QAT-vs-PTQ layer divergence

For each common layer, compare the **op-type composition** (kernel-role counts) of QAT vs PTQ.
- **EXACT** — identical composition (L1 count-distance = 0): explicit Q/DQ changed nothing.
- **PARTIAL** — 1-2 kernel-role mismatches.
- **DIVERGENT** — 3+ mismatches: fusion genuinely differs.
This localises *where* explicit Q/DQ breaks fusion and *how much latency* that costs.

In [ ]:
OPS = sorted(set(DFS['QAT']['op_type']) | set(DFS['PTQ']['op_type']))
def op_count_vec(tag):
    return DFS[tag].groupby(['layer','op_type']).size().unstack(fill_value=0).reindex(columns=OPS, fill_value=0)
def op_ms_vec(tag):
    return DFS[tag].groupby(['layer','op_type'])['medianMs'].sum().unstack(fill_value=0).reindex(columns=OPS, fill_value=0)
Qc, Pc = op_count_vec('QAT'), op_count_vec('PTQ')
Qm, Pm = op_ms_vec('QAT'),   op_ms_vec('PTQ')
common = sorted(set(Qc.index) & set(Pc.index))

rows = []
for L in common:
    qc, pc = Qc.loc[L], Pc.loc[L]
    l1 = int((qc-pc).abs().sum())
    qms, pms = float(Qm.loc[L].sum()), float(Pm.loc[L].sum())
    diffs = {op:int(qc[op]-pc[op]) for op in OPS if qc[op]!=pc[op]}
    match = 'EXACT' if l1==0 else ('PARTIAL' if l1<=2 else 'DIVERGENT')
    rows.append(dict(layer=L, match=match, l1_divergence=l1, QAT_k=int(qc.sum()), PTQ_k=int(pc.sum()),
                     dk=int(qc.sum()-pc.sum()), QAT_ms=round(qms,4), PTQ_ms=round(pms,4),
                     dms=round(qms-pms,4), diffs=diffs))
D = pd.DataFrame(rows).sort_values('l1_divergence', ascending=False).reset_index(drop=True)
D.drop(columns='diffs').to_csv('layer_divergence_QAT_PTQ.csv', index=False)
display(D.drop(columns='diffs'))


## STEP 9a — match-class summary: divergent layers carry the whole QAT-PTQ gap

In [ ]:
summ = D.groupby('match').agg(layers=('layer','size'), QAT_ms=('QAT_ms','sum'),
                              PTQ_ms=('PTQ_ms','sum'), gap_ms=('dms','sum')).reindex(['EXACT','PARTIAL','DIVERGENT'])
display(summ.round(4))
print(f"EXACT-match layers: QAT and PTQ differ by only {summ.loc['EXACT','gap_ms']:+.4f} ms (fusion identical).")
print(f"DIVERGENT layers:   carry {summ.loc['DIVERGENT','gap_ms']:+.4f} ms of the QAT-PTQ gap.")


## STEP 9b — divergence bar (which layers differ most) + match-class latency

In [ ]:
CLSCOL = {'EXACT':'#27ae60','PARTIAL':'#f39c12','DIVERGENT':'#c0392b'}
d = D.sort_values('l1_divergence')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(d['layer'], d['l1_divergence'], color=[CLSCOL[c] for c in d['match']])
ax.set_xlabel('L1 divergence (kernel-role mismatches, QAT vs PTQ)')
ax.set_title('Per-layer QAT-vs-PTQ fusion divergence')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=CLSCOL[c], label=c) for c in ['EXACT','PARTIAL','DIVERGENT']])
plt.tight_layout(); plt.savefig('layer_divergence.png', dpi=130); plt.show()

# match-class summed latency
cls = D.groupby('match')[['QAT_ms','PTQ_ms']].sum().reindex(['EXACT','PARTIAL','DIVERGENT'])
x = np.arange(len(cls)); w=0.38
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(x-w/2, cls['QAT_ms'], w, label='QAT', color='#c0392b')
ax.bar(x+w/2, cls['PTQ_ms'], w, label='PTQ', color='#2980b9')
ax.set_xticks(x); ax.set_xticklabels(cls.index); ax.set_ylabel('summed median ms')
ax.set_title('Summed latency by match-class — QAT vs PTQ')
ax.legend(); plt.tight_layout(); plt.savefig('divergence_by_class.png', dpi=130); plt.show()


## STEP 9c — what op-types drive the divergence (kernel-count deltas across DIVERGENT layers)

In [ ]:
agg = {}
for dd in D[D['match']=='DIVERGENT']['diffs']:
    for op, v in dd.items(): agg[op] = agg.get(op, 0) + v
drv = pd.Series(agg).sort_values()
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.barh(drv.index, drv.values, color=['#c0392b' if v>0 else '#2980b9' for v in drv.values])
for i, v in enumerate(drv.values): ax.text(v + (0.3 if v>=0 else -0.3), i, f'{v:+d}', va='center',
        ha='left' if v>=0 else 'right', fontsize=8)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('kernel-count delta (QAT − PTQ), summed over divergent layers')
ax.set_title('What op-types drive QAT-vs-PTQ divergence')
plt.tight_layout(); plt.savefig('divergence_drivers.png', dpi=130); plt.show()
print('Positive = QAT has MORE of that kernel type (fusion break); negative = PTQ has more.')


## STEP 9d — composition contrast: EXACT-match vs DIVERGENT layers
Stacked op-type composition (kernel counts) for representative layers of each class.

In [ ]:
ex = list(D[D['match']=='EXACT']['layer'])[:3]
dv = list(D[D['match']=='DIVERGENT'].sort_values('l1_divergence', ascending=False)['layer'])
dv = [l for l in dv if l != 'other/glue'][:3]
picks = ex + dv
import matplotlib.cm as cm
palette = {op: cm.tab20(i/len(OPS)) for i, op in enumerate(OPS)}
fig, axes = plt.subplots(1, len(picks), figsize=(3.0*len(picks), 4.6), sharey=True)
for ax, L in zip(axes, picks):
    q = Qc.loc[L]; p = Pc.loc[L]
    bottoms = [0, 0]
    for op in OPS:
        vals = [q[op], p[op]]
        if sum(vals) == 0: continue
        ax.bar(['QAT','PTQ'], vals, bottom=bottoms, color=palette[op], label=op)
        bottoms = [bottoms[0]+vals[0], bottoms[1]+vals[1]]
    cls = D[D['layer']==L]['match'].iloc[0]
    ax.set_title(f'{L}\n({cls})', fontsize=9); ax.set_ylabel('kernel count')
handles, labels = axes[-1].get_legend_handles_labels()
uniq = dict(zip(labels, handles))
fig.legend(uniq.values(), uniq.keys(), loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
fig.suptitle('Kernel-composition: EXACT-match (left) vs DIVERGENT (right) layers', y=1.02)
plt.tight_layout(); plt.savefig('composition_exact_vs_divergent.png', dpi=130, bbox_inches='tight'); plt.show()


## STEP 9e — quantified verdict


In [ ]:
gap_total = D['dms'].sum()
gap_div   = D[D['match']=='DIVERGENT']['dms'].sum()
gap_exact = D[D['match']=='EXACT']['dms'].sum()
n_exact = (D['match']=='EXACT').sum(); n_div = (D['match']=='DIVERGENT').sum()
aq = agg.get('activation quant', 0)
print('QUANTIFIED VERDICT (QAT vs PTQ, per-kernel-profile ms):')
print(f'  {n_exact} EXACT-match layers (linear convs): gap {gap_exact:+.4f} ms  -> Q/DQ changed NOTHING')
print(f'  {n_div} DIVERGENT layers (C2f/attn/head):    gap {gap_div:+.4f} ms  -> the whole difference')
print(f'  total common-layer gap:                       {gap_total:+.4f} ms')
print(f'  divergence driver: +{aq} activation-quant kernels exist in QAT and ZERO in PTQ')
print('  -> explicit Q/DQ is harmless on linear convs; it only breaks fusion at branching layers.')
